In [208]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import operator
import warnings
import time


from sklearn.utils import shuffle
from sklearn import preprocessing
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_score
from sklearn.svm import SVC,LinearSVC
from sklearn import metrics
from sklearn.cluster import KMeans
from imblearn.over_sampling import SMOTE
from scipy.spatial.distance import cdist
from sklearn.preprocessing import StandardScaler


%matplotlib inline
warnings.filterwarnings("ignore")
pd.options.display.max_columns = 200

In [195]:
data = pd.read_csv("./Anuran Calls (MFCCs)/Frogs_MFCCs.csv")
data.head()

,MFCCs_ 1,MFCCs_ 2,MFCCs_ 3,MFCCs_ 4,MFCCs_ 5,MFCCs_ 6,MFCCs_ 7,MFCCs_ 8,MFCCs_ 9,MFCCs_10,MFCCs_11,MFCCs_12,MFCCs_13,MFCCs_14,MFCCs_15,MFCCs_16,MFCCs_17,MFCCs_18,MFCCs_19,MFCCs_20,MFCCs_21,MFCCs_22,Family,Genus,Species,RecordID
0,1.0,0.152936,-0.105586,0.200722,0.317201,0.260764,0.100945,-0.150063,-0.171128,0.124676,0.188654,-0.075622,-0.156436,0.082245,0.135752,-0.024017,-0.108351,-0.077623,-0.009568,0.057684,0.118680,0.014038,Leptodactylidae,Adenomera,AdenomeraAndre,1
1,1.0,0.171534,-0.098975,0.268425,0.338672,0.268353,0.060835,-0.222475,-0.207693,0.170883,0.270958,-0.095004,-0.254341,0.022786,0.163320,0.012022,-0.090974,-0.056510,-0.035303,0.020140,0.082263,0.029056,Leptodactylidae,Adenomera,AdenomeraAndre,1
2,1.0,0.152317,-0.082973,0.287128,0.276014,0.189867,0.008714,-0.242234,-0.219153,0.232538,0.266064,-0.072827,-0.237384,0.050791,0.207338,0.083536,-0.050691,-0.023590,-0.066722,-0.025083,0.099108,0.077162,Leptodactylidae,Adenomera,AdenomeraAndre,1
3,1.0,0.224392,0.118985,0.329432,0.372088,0.361005,0.015501,-0.194347,-0.098181,0.270375,0.267279,-0.162258,-0.317084,-0.011567,0.100413,-0.050224,-0.136009,-0.177037,-0.130498,-0.054766,-0.018691,0.023954,Leptodactylidae,Adenomera,AdenomeraAndre,1
4,1.0,0.087817,-0.068345,0.306967,0.330923,0.249144,0.006884,-0.265423,-0.172700,0.266434,0.332695,-0.100749,-0.298524,0.037439,0.219153,0.062837,-0.048885,-0.053074,-0.088550,-0.031346,0.108610,0.079244,Leptodactylidae,Adenomera,AdenomeraAndre,1


In [196]:
data.shape

(7195, 26)

In [197]:
data["Family"].value_counts(), data["Genus"].value_counts(), data["Species"].value_counts()

(Leptodactylidae    4420
 Hylidae            2165
 Dendrobatidae       542
 Bufonidae            68
 Name: Family, dtype: int64, Adenomera        4150
 Hypsiboas        1593
 Ameerega          542
 Dendropsophus     310
 Leptodactylus     270
 Scinax            148
 Osteocephalus     114
 Rhinella           68
 Name: Genus, dtype: int64, AdenomeraHylaedactylus    3478
 HypsiboasCordobae         1121
 AdenomeraAndre             672
 Ameeregatrivittata         542
 HypsiboasCinerascens       472
 HylaMinuta                 310
 LeptodactylusFuscus        270
 ScinaxRuber                148
 OsteocephalusOophagus      114
 Rhinellagranulosa           68
 Name: Species, dtype: int64)

### 1. Multi-class and Multi-Label Classification Using Support Vector Machines

#### 1(a) Choose 70% of the data randomly as the training set.

In [198]:
data = data.drop(["RecordID"],axis=1)
data = shuffle(data)
features = data.iloc[:,0:-3]
target = data.iloc[:,-3:]
X_train, X_test, y_train, y_test = train_test_split(features, target, test_size = 0.3, random_state = 42)

In [199]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((5036, 22), (2159, 22), (5036, 3), (2159, 3))

#### 1(b) Each instance has three labels: Families, Genus, and Species. Each of the labels has multiple classes. We wish to solve a multi-class and multi-label problem. One of the most important approaches to multi-class classification is to train a classifier for each label. We first try this approach:

##### (i) Research exact match and hamming score/ loss methods for evaluating multilabel classification and use them in evaluating the classifiers in this problem.

##### (ii) Train a SVM for each of the labels, using Gaussian kernels and one versus all classifiers. Determine the weight of the SVM penalty and the width of the Gaussian Kernel using 10 fold cross validation. 1 You are welcome to try to solve the problem with both standardized 2 and raw attributes and report the results.

In [200]:
def encode_labels(y_train):
    encoder = preprocessing.LabelEncoder()
    encoder.fit(y_train)
    y_train = encoder.transform(y_train)
    return y_train

In [201]:
y_train_c0 = encode_labels(y_train.iloc[:,0])
y_train_c1 = encode_labels(y_train.iloc[:,1])
y_train_c2 = encode_labels(y_train.iloc[:,2])

y_test_c0 = encode_labels(y_test.iloc[:,0])
y_test_c1 = encode_labels(y_test.iloc[:,1])
y_test_c2 = encode_labels(y_test.iloc[:,2])

In [219]:
def predict_svm_model(y_train,y_test):
    grid = {'C':[0.001, 0.01, 0.1, 1, 10, 100, 1000],'gamma':[0.001, 0.01, 1, 10, 100]}
    svm_model = GridSearchCV(SVC(kernel='rbf'), grid, cv=10)
    svm_model.fit(X_train, y_train)
    print('Best score for training data:', svm_model.best_score_,"\n") 

    print('Best C:',svm_model.best_estimator_.C,"\n") 
    print('Best Gamma:',svm_model.best_estimator_.gamma,"\n")

    final_model = svm_model.best_estimator_
    y_pred = final_model.predict(X_test)
    loss = metrics.hamming_loss(y_test,y_pred)
    print("Hamming loss :- ",metrics.hamming_loss(y_test,y_pred))
    return loss,y_pred

In [220]:
t = time.time()
loss_c0,y_pred_c0 = predict_svm_model(y_train_c0,y_test_c0)
print("Time taken :- ",time.time()-t)
t = time.time()
loss_c1,y_pred_c1 = predict_svm_model(y_train_c1,y_test_c1)
print("Time taken :- ",time.time()-t)
t = time.time()
loss_c2,y_pred_c2 = predict_svm_model(y_train_c2,y_test_c2)
print("Time taken :- ",time.time()-t)
loss_avg = np.average([loss_c0,loss_c1,loss_c2])

Best score for training data: 0.9920571882446386 

Best C: 10 

Best Gamma: 1 

Hamming loss :-  0.010189902732746642
Time taken :-  293.3299283981323
Best score for training data: 0.9910643367752184 

Best C: 100 

Best Gamma: 1 

Hamming loss :-  0.012042612320518759
Time taken :-  406.43370842933655
Best score for training data: 0.9894757744241461 

Best C: 100 

Best Gamma: 1 

Hamming loss :-  0.01157943492357573
Time taken :-  403.72093391418457


In [239]:
def exact_match(row):
    pred_list = [row['p0'],row['p1'],row['p2']]
    true_list = [row['t0'],row['t1'],row['t2']]
    row['exact_match'] = metrics.accuracy_score(true_list,pred_list)
    return row

In [243]:
print("Average hamming loss :- ",loss_avg)
predicted_labels = pd.concat([pd.Series(y_pred_c0),pd.Series(y_pred_c1),pd.Series(y_pred_c2)],axis = 1)
true_labels = pd.concat([pd.Series(y_test_c0),pd.Series(y_test_c1),pd.Series(y_test_c2)],axis = 1)
compare = pd.DataFrame(pd.concat([predicted_labels,true_labels],axis=1))
compare.columns=["p0","p1","p2","t0","t1","t2"]
compare = compare.apply(exact_match,axis=1)
print("Exact match :- ",np.average(compare["exact_match"]))

Average hamming loss :-  0.011270649992280377
Exact match :-  0.9887293500077198


##### (iii) Repeat 1(b)ii with L 1 -penalized SVMs. 3 Remember to standardize 4 the attributes. Determine the weight of the SVM penalty using 10 fold cross validation.

In [213]:
scaler = StandardScaler()
scaled_X_train = pd.DataFrame(scaler.fit_transform(X_train), columns=list(X_train.columns))
scaled_X_test = pd.DataFrame(scaler.fit_transform(X_test), columns=list(X_test.columns))

In [245]:
def predict_svm_l1_model(X_train,X_test,y_train,y_test):
    # Gaussian kernel
    grid = {'C':[0.001, 0.01, 0.1, 1, 10, 100, 1000]}
    linearsvc_model = GridSearchCV(LinearSVC(penalty = 'l1',dual = False), grid, cv = 10)
    linearsvc_model.fit(X_train, y_train)
    print('Best score for training data:', linearsvc_model.best_score_,"\n") 
    print('Best C:',linearsvc_model.best_estimator_.C,"\n") 

    final_svc_model = linearsvc_model.best_estimator_
    y_pred = final_svc_model.predict(X_test)
    loss = metrics.hamming_loss(y_test,y_pred)
    print("Hamming loss :- ",metrics.hamming_loss(y_test,y_pred))
    return loss,y_pred

In [246]:
t = time.time()
l1_loss_c0,l1_y_pred_c0 = predict_svm_l1_model(scaled_X_train,scaled_X_test,y_train_c0,y_test_c0)
print("Time taken :- ",time.time()-t)
t = time.time()
l1_loss_c1,l1_y_pred_c1 = predict_svm_l1_model(scaled_X_train,scaled_X_test,y_train_c1,y_test_c1)
print("Time taken :- ",time.time()-t)
t = time.time()
l1_loss_c2,l1_y_pred_c2 = predict_svm_l1_model(scaled_X_train,scaled_X_test,y_train_c2,y_test_c2)
print("Time taken :- ",time.time()-t)

Best score for training data: 0.9382446386020651 

Best C: 1 

Hamming loss :-  0.06901343214451135
Time taken :-  79.56958055496216
Best score for training data: 0.9517474185861795 

Best C: 1000 

Hamming loss :-  0.052802223251505326
Time taken :-  113.9242684841156
Best score for training data: 0.9571088165210484 

Best C: 1 

Hamming loss :-  0.04029643353404354
Time taken :-  109.46533584594727


In [247]:
l1_loss_avg = np.average([l1_loss_c0,l1_loss_c1,l1_loss_c2])
print("Average hamming loss :- ",l1_loss_avg)
l1_predicted_labels = pd.concat([pd.Series(l1_y_pred_c0),pd.Series(l1_y_pred_c1),pd.Series(l1_y_pred_c2)],axis = 1)
l1_true_labels = pd.concat([pd.Series(y_test_c0),pd.Series(y_test_c1),pd.Series(y_test_c2)],axis = 1)
l1_compare = pd.DataFrame(pd.concat([l1_predicted_labels,l1_true_labels],axis=1))
l1_compare.columns=["p0","p1","p2","t0","t1","t2"]
l1_compare = l1_compare.apply(exact_match,axis=1)
print("Exact match :- ",np.average(l1_compare["exact_match"]))

Average hamming loss :-  0.05403736297668674
Exact match :-  0.9459626370233132


##### (iv) Repeat 1(b)iii by using SMOTE or any other method you know to remedy class imbalance. Report your conclusions about the classifiers you trained.

In [217]:
sm = SMOTE(random_state = 42)
X_bal_c0, y_bal_c0 = sm.fit_resample(X_train, y_train_c0)
X_bal_c1, y_bal_c1 = sm.fit_resample(X_train, y_train_c1)
X_bal_c2, y_bal_c2 = sm.fit_resample(X_train, y_train_c2)

In [248]:
t = time.time()
bal_loss_c0,bal_y_pred_c0 = predict_svm_l1_model(X_bal_c0,X_test,y_bal_c0,y_test_c0)
print("Time taken :- ",time.time()-t)
t = time.time()
bal_loss_c1,bal_y_pred_c1 = predict_svm_l1_model(X_bal_c1,X_test,y_bal_c1,y_test_c1)
print("Time taken :- ",time.time()-t)
t = time.time()
bal_loss_c2,bal_y_pred_c2 = predict_svm_l1_model(X_bal_c2,X_test,y_bal_c2,y_test_c2)
print("Time taken :- ",time.time()-t)

Best score for training data: 0.9488636363636364 

Best C: 1 

Hamming loss :-  0.08198239925891616
Time taken :-  325.99147295951843
Best score for training data: 0.9531087634222376 

Best C: 1000 

Hamming loss :-  0.07688744789254284
Time taken :-  6559.684553384781
Best score for training data: 0.9608551266085512 

Best C: 100 

Hamming loss :-  0.041222788327929596
Time taken :-  10256.277529478073


In [249]:
bal_loss_avg = np.average([bal_loss_c0,bal_loss_c1,bal_loss_c2])
print("Average hamming loss :- ",bal_loss_avg)
bal_predicted_labels = pd.concat([pd.Series(bal_y_pred_c0),pd.Series(bal_y_pred_c1),pd.Series(bal_y_pred_c2)],axis = 1)
bal_true_labels = pd.concat([pd.Series(y_test_c0),pd.Series(y_test_c1),pd.Series(y_test_c2)],axis = 1)
bal_compare = pd.DataFrame(pd.concat([bal_predicted_labels,bal_true_labels],axis=1))
bal_compare.columns=["p0","p1","p2","t0","t1","t2"]
bal_compare = bal_compare.apply(exact_match,axis=1)
print("Exact match :- ",np.average(bal_compare["exact_match"]))

Average hamming loss :-  0.0666975451597962
Exact match :-  0.9333024548402038


#### Hamming loss of the gaussian kernel is less than that of the linear kernel which is lesser than the balanced dataset. Exact match decreases from 1b(ii) to 1b(iii) to 1b(iv)

### 2. K-Means Clustering on a Multi-Class and Multi-Label Data Set

#### Monte-Carlo Simulation: Perform the following procedures 50 times, and report the average and standard deviation of the 50 Hamming Distances that you calculate.

##### (a) Use k-means clustering on the whole Anuran Calls (MFCCs) Data Set (do not split the data into train and test, as we are not performing supervised learning in this exercise). Choose k ∈ {1, 2, . . . , 50} automatically based on one of the methods provided in the slides (CH or Gap Statistics or scree plots or Silhouettes) or any other method you know.

##### (b) In each cluster, determine which family is the majority by reading the true labels.Repeat for genus and species.

##### (c) Now for each cluster you have a majority label triplet (family, genus, species). Calculate the average Hamming distance, Hamming score, and Hamming loss 5 between the true labels and the labels assigned by clusters

In [161]:
data = shuffle(data)
target_encoded = pd.concat([pd.Series(encode_labels(target.iloc[:,0])), \
                            pd.Series(encode_labels(target.iloc[:,1])), \
                            pd.Series(encode_labels(target.iloc[:,2]))],axis=1)

data = pd.concat([features, target_encoded],axis=1)
new_cols  = list(data.columns[0:22])
new_cols.append("Family")
new_cols.append("Genus")
new_cols.append("Species")
data.columns = new_cols

In [147]:
def add_hamming_distance(row):
    row["true_labels"] = tuple(int(x) for x in row[22:25])
    row["assigned_labels"] = tuple(per_cluster_majority[row['cluster_index']].values())
    row["hamming_distance"] = sum(el1 != el2 for el1, el2 in zip(row["true_labels"], row["assigned_labels"]))
    return row

In [148]:
clustered_data = clustered_data.apply(add_hamming_distance,axis=1)

In [150]:
def calculate_hamming_loss_score(row):
    row["hamming_loss"] = metrics.hamming_loss(row["true_labels"],row["assigned_labels"])
    row["hamming_score"] = 1 - row["hamming_loss"]
    return row

In [185]:
num_simulations = 50
avgs = []
k_range = list(range(1,51))

for n in range(0,num_simulations):
    silhouette_scores = []
    pred_list = []
    for k in k_range[1:]:
        clusterer = KMeans (n_clusters = k)
        preds = clusterer.fit_predict(features)
        pred_list.append(preds)
        score = metrics.silhouette_score(features,preds, metric = 'euclidean')
        #print ("For n_clusters = {}, silhouette score is {}".format(k, score))
        silhouette_scores.append(score)
        
    best_k = silhouette_scores.index(max(silhouette_scores)) + 2
    cluster_index = pd.Series(pred_list[silhouette_scores.index(max(silhouette_scores))])
    clustered_data = pd.concat([data,cluster_index],axis=1)
    new_cols = list(clustered_data.columns)
    #print(new_cols)
    new_cols[-1] = 'cluster_index'
    clustered_data.columns = new_cols

    per_cluster_majority = {}
    for i in range(0,best_k):
        per_cluster_majority[i] = {}
        for j in ['Family','Genus','Species']:
            per_cluster_majority[i][j] = clustered_data[clustered_data['cluster_index'] == i][j].value_counts().index[0]

    clustered_data = clustered_data.apply(add_hamming_distance,axis=1)
    clustered_data = clustered_data.apply(calculate_hamming_loss_score,axis=1)
    avgs.append([np.average(clustered_data["hamming_distance"]),np.average(clustered_data["hamming_loss"]),np.average(clustered_data["hamming_score"])])
    print("Simulation No. ",n, " done")        

Simulation No.  0  done
Simulation No.  1  done
Simulation No.  2  done
Simulation No.  3  done
Simulation No.  4  done
Simulation No.  5  done
Simulation No.  6  done
Simulation No.  7  done
Simulation No.  8  done
Simulation No.  9  done
Simulation No.  10  done
Simulation No.  11  done
Simulation No.  12  done
Simulation No.  13  done
Simulation No.  14  done
Simulation No.  15  done
Simulation No.  16  done
Simulation No.  17  done
Simulation No.  18  done
Simulation No.  19  done
Simulation No.  20  done
Simulation No.  21  done
Simulation No.  22  done
Simulation No.  23  done
Simulation No.  24  done
Simulation No.  25  done
Simulation No.  26  done
Simulation No.  27  done
Simulation No.  28  done
Simulation No.  29  done
Simulation No.  30  done
Simulation No.  31  done
Simulation No.  32  done
Simulation No.  33  done
Simulation No.  34  done
Simulation No.  35  done
Simulation No.  36  done
Simulation No.  37  done
Simulation No.  38  done
Simulation No.  39  done
Simulation

In [186]:
avgs

[[0.66726893676164, 0.22242297892054666, 0.7775770210794534],
 [0.66726893676164, 0.22242297892054666, 0.7775770210794534],
 [0.66726893676164, 0.22242297892054666, 0.7775770210794534],
 [0.6674079221681724, 0.22246930738939075, 0.7775306926106093],
 [0.735371785962474, 0.24512392865415797, 0.7548760713458421],
 [0.7357887421820709, 0.24526291406069028, 0.7547370859393098],
 [0.66726893676164, 0.22242297892054666, 0.7775770210794534],
 [0.66726893676164, 0.22242297892054666, 0.7775770210794534],
 [0.66726893676164, 0.22242297892054666, 0.7775770210794534],
 [0.66726893676164, 0.22242297892054666, 0.7775770210794534],
 [0.66726893676164, 0.22242297892054666, 0.7775770210794534],
 [0.66726893676164, 0.22242297892054666, 0.7775770210794534],
 [0.6669909659485754, 0.2223303219828585, 0.7776696780171416],
 [0.66726893676164, 0.22242297892054666, 0.7775770210794534],
 [0.66726893676164, 0.22242297892054666, 0.7775770210794534],
 [0.735371785962474, 0.24512392865415797, 0.7548760713458421],
 

In [187]:
saved = avgs

In [193]:
mat = pd.DataFrame(avgs)
print("Average of the hamming distances of 50 simulations :- ",np.average(mat[0]))
print("Standard Deviation of the hamming distances of 50 simulations :- ",np.std(mat[0]))

Average of the hamming distances of 50 simulations :-  0.6739624739402362
Standard Deviation of the hamming distances of 50 simulations :-  0.020501027962463277


In [154]:
clustered_data.head()

,MFCCs_ 1,MFCCs_ 2,MFCCs_ 3,MFCCs_ 4,MFCCs_ 5,MFCCs_ 6,MFCCs_ 7,MFCCs_ 8,MFCCs_ 9,MFCCs_10,MFCCs_11,MFCCs_12,MFCCs_13,MFCCs_14,MFCCs_15,MFCCs_16,MFCCs_17,MFCCs_18,MFCCs_19,MFCCs_20,MFCCs_21,MFCCs_22,Family,Genus,Species,cluster_index,true_labels,assigned_labels,hamming_distance,hamming_loss,hamming_score
0,1.0,0.152936,-0.105586,0.200722,0.317201,0.260764,0.100945,-0.150063,-0.171128,0.124676,0.188654,-0.075622,-0.156436,0.082245,0.135752,-0.024017,-0.108351,-0.077623,-0.009568,0.057684,0.118680,0.014038,2.0,3.0,5.0,0.0,"(2, 3, 5)","(2, 3, 5)",0,0.0,1.0
1,1.0,0.171534,-0.098975,0.268425,0.338672,0.268353,0.060835,-0.222475,-0.207693,0.170883,0.270958,-0.095004,-0.254341,0.022786,0.163320,0.012022,-0.090974,-0.056510,-0.035303,0.020140,0.082263,0.029056,3.0,4.0,6.0,0.0,"(3, 4, 6)","(2, 3, 5)",3,1.0,0.0
2,1.0,0.152317,-0.082973,0.287128,0.276014,0.189867,0.008714,-0.242234,-0.219153,0.232538,0.266064,-0.072827,-0.237384,0.050791,0.207338,0.083536,-0.050691,-0.023590,-0.066722,-0.025083,0.099108,0.077162,3.0,0.0,1.0,1.0,"(3, 0, 1)","(3, 0, 1)",0,0.0,1.0
3,1.0,0.224392,0.118985,0.329432,0.372088,0.361005,0.015501,-0.194347,-0.098181,0.270375,0.267279,-0.162258,-0.317084,-0.011567,0.100413,-0.050224,-0.136009,-0.177037,-0.130498,-0.054766,-0.018691,0.023954,2.0,3.0,5.0,0.0,"(2, 3, 5)","(2, 3, 5)",0,0.0,1.0
4,1.0,0.087817,-0.068345,0.306967,0.330923,0.249144,0.006884,-0.265423,-0.172700,0.266434,0.332695,-0.100749,-0.298524,0.037439,0.219153,0.062837,-0.048885,-0.053074,-0.088550,-0.031346,0.108610,0.079244,2.0,2.0,3.0,2.0,"(2, 2, 3)","(1, 1, 2)",3,1.0,0.0


In [ ]:
# Citation
# https://www.kagglpde.com/pranathichunduru/svm-for-multiclass-classification/code
# https://pythonprogramminglanguage.com/kmeans-elbow-method/
# https://stackoverflow.com/questions/51138686/how-to-use-silhouette-score-in-k-means-clustering-from-sklearn-library